In [ ]:
# Demo 4: Test claim extraction with cleaned/chunked transcript (start video batching)

# Use to get hugging face models
!pip install transformers accelerate huggingface_hub
# Use python library to get youtube transcript
!pip install youtube-transcript-api

In [ ]:
# Libraries
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline

"""
Integrate youtube_api.py code:

import os
from dotenv import load_dotenv
from googleapiclient.discovery import build
import pandas as pd
"""

In [ ]:
"""
Integrate youtube_api.py code:

load_dotenv()

api_key= os.getenv("API_KEY")

yta = YouTubeTranscriptApi()


def get_transcript(video_id):
    try:
        transcript = yta.fetch(video_id)
        transcript_text = " ".join(entry.text for entry in transcript)
        return transcript_text
    except Exception:
        return None


youtube = build('youtube', 'v3', developerKey=api_key)

request = youtube.videos().list(
    part='localizations,statistics,topicDetails',
    chart='mostPopular',
    regionCode='us',
    videoCategoryId='20',
    maxResults='50'
)

response = request.execute()

videos_data = []

for item in response["items"]:
    video_id = item.get("id")

    videos_data.append({"Video ID": video_id})

df = pd.DataFrame(videos_data)
# df["Transcript"] = df["Video ID"].apply(get_transcript)
"""

In [ ]:
# Load Qwen model (small free version)
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype="auto",
    device_map="auto",
    max_new_tokens=200 # increase to prevent incomplete output
)

In [ ]:
"""
# Test retrieve youtube video transcript (for 1 video)

video_id = "TT4HTDYhiOU" # sample video ID
yta = YouTubeTranscriptApi()
transcript = yta.fetch(video_id)
print(f"Transcript:\n{transcript}")
"""


Output Format:

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='my little brother recently got Minecraft', start=0.04, duration=3.639), ...

Format Analysis:

- FetchedTranscript object (holds snippets list)
  - snippets list (holds FetchedTranscriptSnippet objects (each object is a snippet))
    - FetchedTranscriptSnippet object (holds 3 attributes: text, start, duration)

In [6]:
# Test clean raw transcript

def clean_transcript(transcript):
  cleaned = [] # list for cleaned snippets

  # Loop to get text (clean) & get start time
  for snippet in transcript.snippets:
    text = snippet.text # get text from snippet
    start_time = snippet.start # get start time from snippet

    while "[" in text:
      start = text.index("[")
      end = text.index("]") + 1
      caption = text[start:end]
      text = text.replace(caption, "")

    cleaned.append({"text": text, "start": start_time}) # add snippet's text and start time to cleaned list

  # Test print cleaned list
  # print(f"Transcript Text:\n{cleaned}")

  return cleaned

Get cleaned transcript text & start time for each snippet

Transcript Text:

[{'text': 'So guys, Minecraft 26.0 is finally out,', 'start': 0.08}, {'text': 'and this update brings more than 50', 'start': 3.2},...]

In [7]:
# Test chunking cleaned transcript

def chunk_transcript(cleaned):
  chunk_size = 600 # chunk size based on time (sec)
  chunks = [] # list for chunks
  chunk_snippets = [] # list to build each chunk
  curr_chunk_time = 0

  # Loop through all snippets in cleaned[]
  for snippet in cleaned:
      text = snippet["text"] # get snippet text
      chunk_snippets.append(text) # add text to chunk_snippets list

      # Check if reach chunk_size (approx.)
      if snippet["start"] - curr_chunk_time >= chunk_size:
        # print(f"Chunk time: {snippet["start"] - curr_chunk_time} sec") # Test print chunk start time
        chunk = " ".join(chunk_snippets)  # join chunk snippet texts into a string
        chunks.append(chunk) # add chunk to list of chunks

        chunk_snippets = [] # reset chunk_snippets list
        curr_chunk_time = snippet["start"] # reset time to current start time

  # If leftover snippets, create and add last chunk
  if chunk_snippets:
    chunk = " ".join(chunk_snippets)  # join chunk snippet texts into a string
    chunks.append(chunk) # add chunk to list of chunks

  # Test print chunks list
  # print(f"Transcript Text:\n{chunks}")

  return chunks


Test chunk time:(e.g. chunk size = 60 sec)

Chunk time: 60.64 sec

Chunk time: 61.28 sec

Chunk time: 61.599000000000004 sec

Chunk time: 61.601 sec

Chunk time: <60 sec

In [8]:
# Test cleaned/chunked transcript in claim extraction step

# Function to extract claims
def claim_extraction(chunks):
  results = []
  # Get claims for each chunk
  for chunk in chunks:
    # Create message
    messages = [
      {"role": "system", "content": "You are a helpful assistant that extracts factual claims from text."},
      {"role": "user", "content": f"Extract all factual claims as bullet points.\n\nText: {chunk}"}
    ]

    # Send message to Qwen model
    output = pipe(messages, max_length=None)
    # Get claims from Qwen response
    result = output[0]["generated_text"][-1]["content"]
    results.append(result) # add claims to results list
  return results


In [ ]:
# Test extract claim from multiple videos


# video_ids = ["-1Sy6iz43vg", "TT4HTDYhiOU"] # list of short sample video ids
video_ids = ["Pj3WI5kL9aM"] # Test video id from Top 50 video excel sheet (3hr vs 20 min video)

# video_ids = df["Video ID"].tolist() # 50 video ids from df into a list

# Loop through each video id in list
for video_id in video_ids:
# for video_id in video_ids:
  yta = YouTubeTranscriptApi()
  try:
    transcript = yta.fetch(video_id) # get transcript
    chunks = chunk_transcript(clean_transcript(transcript)) # clean/chunk transcript

    # Extract claims from chunks list
    claims_list = claim_extraction(chunks)
    for claims in claims_list:
      print(f"Claims Extracted:\n{claims}\n")
  except Exception: # error for no transcript
    print(f"No transcript for video ID: {video_id}")



Top 50 game videos excel sheet : 3hr, 1.5hr, 16min, 30 min, 1hr 20min, 17min...

Chunk size 300s (approx. 20 min video)- specific

Claims Extracted:
* FATEKEEPER - A magical knight game with a talking robot companion.
* WINDROSE - A pirate-themed survival game with impressive visuals and features.
* FOREST 3 - Another narrative-first person survival game with a focus on exploration and resource management.
* TOMB RAIDER CATALYST AND LEGACY OF ATLANTIS - A remake of the first Tomb Raider game and a full sequel to Underworld, offering a reboot of the franchise.
* RAGDOLL PHYSICS - A game featuring ragdoll mechanics, likely a successor to Dark Messiah of Might and Magic.

Claims Extracted:
* Raiders are made by Crystal Dynamics.
* Crystal Dynamics can produce good games even under poor conditions.
* Total War Warhammer 40K is a revolutionary Total War game focusing on gun-based warfare.
* Lego Batman Legacy of the Dark Knight combines multiple elements of the Lego Batman franchise in a single experience.
* Forta Horizon 6 offers a variety of locations and is known for its reliability and enjoyment.
* Control Resonant is described as a sequel to Control, featuring a semi-open environment and unconventional gameplay mechanics.

Claims Extracted:
- Become a member of the 1% club to listen to the Falcon Show exclusively every Saturday.
- Number 12 is on toast.
- The developers of Soma are behind the iconic Amnesia games.
- The Amnesia games expanded into different strange territory after Amnesia: The Dark Descent.
- The Amnesia team made Soma as well.
- Antas or Antos is a game that is exciting because of its pedigree.
- Actor still in Scarsgard is voicing a prominent character in Bradley the Badger.
- Bradley the Badger is described as a throwback to '90s 3D platformers.
- Gang of Dragon is expected to feature more blood, gun violence, and general crime than previous Yakaza adventures.
- Ace Combat 8: Wings of Thieve is set in an alternate reality military simulation and allows players to engage in conflicts without real-life baggage.
- Mega Man Dual Override continues to impress fans of the franchise.

Claims Extracted:
- **Mega Man 12** is expected to come out in 2027.
- **Galactic Racer** returns after a long absence; it features land-speeders, speeder bikes, and skim speeders, with a single-player narrative.
- **Castlevania Belmont's Curse** returns as a sidescrolling platforming game.
- **Tropico 7** is described as "guaranteed to be good" due to the expertise of the developers behind previous titles.
- **Warhammer 40k Dawn of War 4** promises nostalgic favorites with new factions and improved controls for the base-building aspect.

Claims Extracted:
- Switching developers can be seen as a negative sign for two and three.
- King Art is aware of their previous projects' reception when making two and three.
- The author wishes "number three" would perform well.
- "Darkwood 2" is a sequel from developers who previously worked on the Pathologic series.
- "Darkwood 2" is described as psychologically twisted.
- Survival games often include elements such as time limits, scavenging during the day, and hiding at night.
- "Darkwood 2" features an environment where armed soldiers patrol the area.
- The setting of "Darkwood 2" differs from "Darkwood."
- The main character of "Darkwood 2" must construct a giant train to progress through the game.
- "Railorn" offers a unique gameplay mechanic where players need to construct a giant train to survive.
- Players in "Railorn" start with a simple handcart and gradually upgrade to more advanced locomotives


Chunk size 600s (approx. 20 min video)- generic

Claims Extracted:
* Ubisoft, Microsoft, and Sony are experiencing a transitional period.
* There are numerous upcoming games to choose from.
* Games like "Fatekeeper" and "Wind Rose" are highly anticipated.
* "Fatekeeper" features a magical knight with a talking rack companion and giant sword.
* "Wind Rose" is a survival game inspired by pirates and adventure.
* "The Forest 3" is a narrative-first person survival game.
* "Tomb Raider Catalyst and Legacy of Atlantis" is a remake and sequel.
* "Total War Warhammer 40k" is a new historical RTS game.
* "Lego Batman Legacy of the Dark Knight" is a Lego adaptation of Batman comics.
* "Forta Horizon 6" is a reimagined racing game.
* "Control Resonant" is a semi-sequel to "Control."

Claims Extracted:
- Become a member of the 1% club to listen to the Falcon Show exclusively every Saturday.
- Amnesia: The Dark Descent is a horror classic due to its development team behind the iconic Amnesia games.
- Antas or Antos may have explicit connections to Soma.
- Bradley the Badger is a surprising addition to the gaming community, voiced by an actor known for roles in "Andor" and "Chernobyl."
- Gang of Dragon is a mascot platformer inspired by '90s 3D platformers, promising intense gameplay elements such as brawling, reckless driving, and gun violence.
- Ace Combat 8: Wings of Thieve is an arcade action game featuring modern fighter jets and engaging narratives.
- Mega Man Dual Override returns to the beloved Mega Man franchise, offering a sidescrolling action experience.
- Star Wars Galactic Racer is a Star Wars-themed racing game, bringing nostalgia and fresh mechanics to the genre.
- Castlevania

Claims Extracted:
* Very high quality RTS that is worth checking out on Steam if you're a fan of this genre of game.
* King Art knows what the hell they're doing.
* Awarded "wish list" status for its development team.
* Darkwood 2 is a sequel from the developers of the Pathologic series, aiming to deliver an original experience despite coming from a completely different background.
* Darkwood is a psychological horror game with a unique setting and tone, likely challenging expectations.
* The new developers face the challenge of maintaining the same standards while bringing a fresh perspective to a beloved franchise.
* Railorn combines survival mechanics with a unique gameplay mechanic of constructing a giant train through various upgrades and challenges.
* Divinity is expected to be a lavish, detailed post-apocalyptic survival game, possibly addressing themes of nostalgia and grandiosity.
* Balders's Gate 3 is highly anticipated, featuring innovative storytelling and production values distinct from traditional RPGs.